# Quickstart

This notebook takes pose data through the whole pipeline and, along the way,
teaches you to **read the checkpoints** and **choose the two embedding parameters
the analysis needs**. Run it top to bottom.

It runs on a small **bundled synthetic recording first**, so you can see everything
work before your own data can confuse matters. When you understand the checkpoints,
swap in your own file at the one marked cell.

You don't need to write code — just run each cell and read the plot above the
prose that explains it.

In [ ]:
from pose_dynamics import load_pose_csv
from pose_dynamics.data import example_fixture
from pose_dynamics.preprocessing import (mask_low_confidence, interpolate_gaps,
    butterworth_filter, assess_quality,
    plot_masking_checkpoint, plot_interpolation_checkpoint, plot_filter_checkpoint)
import numpy as np, matplotlib.pyplot as plt

## Your data goes here

The cell below chooses the input file and its frame rate. It uses the **bundled
example** now. When you're ready, comment out the first line, uncomment the second,
and point it at your own [canonical CSV](../docs/canonical_format.md). The frame
rate is required — the file has no timestamps.

In [ ]:
DATA = example_fixture()          # <-- the bundled synthetic example
# DATA = "/work/data/your_trial.csv"  # <-- SWAP: your file in the mounted data/ folder
#                                     #     (or any path if you pip-installed)
FRAME_RATE = 60.0                  # <-- Hz, must match how your data was recorded

seq = load_pose_csv(DATA, frame_rate=FRAME_RATE)
seq.summary()

**What loaded.** `dims` (2 or 3) and whether there's confidence were inferred
from the column names — you didn't configure them. If loading failed, the error
message names the offending columns; see the [format spec](../docs/canonical_format.md).

## Checkpoint 1 — missing data

Real pose data has gaps: occlusions, low-confidence frames. First we mark
low-confidence keypoints as missing. The plot shows the raw signal with the masked
points in red.

**What "good" looks like:** a handful of short red stretches, not most of the
signal. If red dominates, your confidence threshold is too high, or that keypoint
is poorly tracked and should be dropped rather than analysed.

In [ ]:
masked = mask_low_confidence(seq, threshold=0.30)
plot_masking_checkpoint(seq, masked, keypoint=0);

## Checkpoint 2 — interpolation

Short gaps are filled by drawing a straight line across them; gaps longer than the
cap are **left missing on purpose** — filling a long gap invents movement that never
happened. Green marks filled samples; red shaded spans are gaps left alone.

**What "good" looks like:** green fills on the brief gaps, and any long dropout left
red. The note prints how many samples were filled versus left missing.

In [ ]:
interp = interpolate_gaps(masked, max_gap=60)   # 60 frames = 1 s at 60 Hz
print(interp.provenance[-1].note)
plot_interpolation_checkpoint(masked, interp, keypoint=2);   # kp2 has a long gap

## Checkpoint 3 — filtering

A low-pass filter removes frame-to-frame jitter. The plot overlays the signal before
(grey) and after (orange).

**What "good" looks like:** the orange line tracks the *shape* of the movement and
removes the fuzz — it should **smooth, not flatten**. If the orange line has lost the
peaks and troughs of the real movement, your cutoff is too low.

In [ ]:
filt = butterworth_filter(interp, cutoff_hz=10.0, order=4)
fig, ax = plt.subplots(figsize=(11, 3.5))
plot_filter_checkpoint(interp, filt, keypoint=0, ax=ax); ax.set_xlim(20, 30);

A quick data-quality read on the whole trial — status and the largest gap that
survived interpolation. Nothing is discarded silently: flagged/excluded trials show
up here.

In [ ]:
assess_quality(interp, on_exceed="flag").summary()

## Movement magnitude — linear metrics

Before any recurrence analysis, you can already answer *how much* and *how fast*
people move. This is a complete analysis on its own — many questions need nothing
more. The table below summarises each keypoint's displacement, speed, and
acceleration (mean, RMS, max).

The paper pairs this **magnitude** view with the **organization** view that
recurrence adds next — two signals can have identical amplitude statistics yet
completely different temporal structure.

In [ ]:
from pose_dynamics.linear import kinematic_summary

kinematic_summary(filt, stats=("mean", "rms", "max")).round(2)

## The decision — choosing (τ, m)

Recurrence analysis reconstructs the movement's *state space* using two numbers: a
**delay τ** and a **dimension m**. The framework will not pick them for you — the
paper argues automated picking is unreliable. Instead it computes the evidence
across all your signals and **proposes** values; **you** read the plots and commit.

Below, `select_embedding` runs Average Mutual Information (AMI, for τ) and False
Nearest Neighbours (FNN, for m) on every keypoint channel.

In [ ]:
from pose_dynamics.embedding import select_embedding, coordinate_channels, plot_embedding_evidence

evidence = select_embedding(coordinate_channels(filt),
                            tau_grid=(10, 25), m_grid=(3, 6),
                            ami_max_lag=50, fnn_max_dim=8)
plot_embedding_evidence(evidence)
print(evidence.justification)

### How to read these two plots

**Left (AMI → τ).** Each faint line is one signal; the bold line is the median, the
band its spread. Read the **shape**:

- The curve drops steeply, then flattens. **Pick τ near where it stops dropping** —
  the *onset of the plateau*. Beyond there, more delay buys little new information.
- A clear dip (a first local minimum) is the textbook case; τ at the dip is fine.
- **Plateau vs. noise — the judgement that matters:** a *real* minimum is a broad
  turn that shows up across many of the faint lines. A one-sample jiggle in a single
  curve is noise — ignore it. The proposal already smooths and looks for a turn
  that's shared, which is why it may differ from the lowest single point.

**Right (FNN → m).** The curve is the percentage of "false" neighbours as you add
dimensions. It drops steeply, then flattens — **pick m at the elbow.** It often
flattens at a *non-zero floor* (measurement noise keeps a few percent false at every
dimension); **don't chase it to zero** — take the knee. Over-shooting m slightly is
safer than under-shooting.

The red dashed lines are the proposal; the grey band is the plausible grid it was
clamped to. If the curves are a shapeless mess with no plateau, your signal is
probably oversampled (downsample and retry) or too noisy to embed.

### Commit

Having read the plots, commit the values. `commit` records your choice (and warns —
without blocking — if you go outside the presented grid or below the proposed
dimension). In a study you would apply this one `(τ, m)` to every trial.

In [ ]:
params = evidence.commit(tau=evidence.proposed_tau, m=evidence.proposed_m)
params.to_dict()

## The result — a recurrence plot

Now we run the recurrence analysis on one signal and draw its **recurrence plot** —
a map of every moment the movement returned to a state it had visited before.

In [ ]:
from pose_dynamics.rqa import RqaParams, run_auto_rqa

signal = filt.coords[:, 0, 0]            # keypoint 0, x — one feature to analyse
signal = signal[np.isfinite(signal)]
rqa = RqaParams(eDim=params.m, tLag=params.tau, radius_mode="fixed_rrec",
                target_rec=5.0, rescale="mean", norm="zscore", min_line=2)
result = run_auto_rqa(signal, rqa)
print(f"%REC={result.rec_rate:.2f}  %DET={result.metrics['perc_determ']:.1f}  "
      f"radius={result.radius_used:.3f}")
result.plot();

**How to read it.** Points on the diagonal are trivial (every state matches
itself). **Diagonal lines off the main diagonal** mean the movement repeated a whole
sequence — rhythm, predictability (that's what *determinism* counts). **Solid
squares/blocks** mean the movement held still (laminar phases). A scattered dust of
isolated points with no lines means noise-like, unpredictable movement. The metrics
above the plot quantify exactly this.

Because we used `fixed_rrec` (a target recurrence rate), the **radius** was solved to
hit that rate and is reported — under this mode the radius, not %REC, is the
informative density measure.

## Where to go next

- **Run your whole dataset** — once you've committed (τ, m), open
  `run_dataset.ipynb`: point it at a folder of canonical CSVs and get a tidy table
  of linear + recurrence metrics out (edit a config, not code).
- **Bring your own data** — convert it to the
  [canonical format](../docs/canonical_format.md) first; see `examples/` for converters.
- **Tune the run** — every parameter, its default, and the paper's guidance is in the
  [configuration reference](../docs/configuration.md).
- **Build a feature pipeline** — compose the [primitives](../docs/feature_steps.md) for
  your own features (apertures, ROIs, kinematics), or reproduce a case study with the
  `notebooks/case*_*.ipynb` notebooks.